In [1]:
import re
import matplotlib.pyplot as plt

In [2]:
from tokenizer import A1Tokenizer, PATTERN
from collections import Counter
import torch

tokenizer = A1Tokenizer.from_file("tokenizers/tokenizer_16384.pkl")

print("Vocabulary size:", len(tokenizer))
print("Number of merge rules:", len(tokenizer.merge_rules))
print("Max model length:", tokenizer.model_max_length)

print("\nSpecial token IDs:")
print("PAD:", tokenizer.pad_token_id)
print("UNK:", tokenizer.unk_token_id)
print("BOS:", tokenizer.bos_token_id)
print("EOS:", tokenizer.eos_token_id)

Vocabulary size: 16384
Number of merge rules: 16124
Max model length: 256

Special token IDs:
PAD: 0
UNK: 1
BOS: 2
EOS: 3


In [3]:
print("First 20 merge rules:\n")

for rank, pair in enumerate(tokenizer.merge_rules[:20]):
    print(
        f"{rank:4d}: {pair[0]!r} + {pair[1]!r} "
        f"-> {(pair[0] + pair[1])!r}"
    )

print("\nLast 20 merge rules:\n")

start = max(0, len(tokenizer.merge_rules) - 20)

for rank in range(start, len(tokenizer.merge_rules)):
    pair = tokenizer.merge_rules[rank]

    print(
        f"{rank:4d}: {pair[0]!r} + {pair[1]!r} "
        f"-> {(pair[0] + pair[1])!r}"
    )

First 20 merge rules:

   0: b' ' + b't' -> b' t'
   1: b'h' + b'e' -> b'he'
   2: b' ' + b'a' -> b' a'
   3: b'i' + b'n' -> b'in'
   4: b' t' + b'he' -> b' the'
   5: b'e' + b'r' -> b'er'
   6: b'o' + b'n' -> b'on'
   7: b'r' + b'e' -> b're'
   8: b' ' + b'o' -> b' o'
   9: b'a' + b't' -> b'at'
  10: b' ' + b's' -> b' s'
  11: b'e' + b'n' -> b'en'
  12: b'e' + b'd' -> b'ed'
  13: b'a' + b'n' -> b'an'
  14: b' ' + b'c' -> b' c'
  15: b'i' + b's' -> b'is'
  16: b' ' + b'w' -> b' w'
  17: b'i' + b't' -> b'it'
  18: b'e' + b's' -> b'es'
  19: b'o' + b'r' -> b'or'

Last 20 merge rules:

16104: b' exter' + b'min' -> b' extermin'
16105: b' B' + b'ordeaux' -> b' Bordeaux'
16106: b' J' + b'oshua' -> b' Joshua'
16107: b'C' + b'atal' -> b'Catal'
16108: b'B' + b'arn' -> b'Barn'
16109: b' \xc3' + b'\x81' -> b' \xc3\x81'
16110: b' el' + b'ves' -> b' elves'
16111: b' Charl' + b'ton' -> b' Charlton'
16112: b'Ec' + b'uador' -> b'Ecuador'
16113: b' sens' + b'ation' -> b' sensation'
16114: b' meaning' +

In [4]:
text = "She lives in Göteborg, Sweden in 2026!"

chunks = tokenizer.pattern.findall(text.strip())

print("Original:", repr(text))
print("\nPre-tokenized chunks:")

for i, chunk in enumerate(chunks):
    print(f"{i:2d}: {chunk!r}")

print("\nReconstruction:", repr("".join(chunks)))
assert "".join(chunks) == text.strip()

Original: 'She lives in Göteborg, Sweden in 2026!'

Pre-tokenized chunks:
 0: 'She'
 1: ' lives'
 2: ' in'
 3: ' Göteborg'
 4: ','
 5: ' Sweden'
 6: ' in'
 7: ' 2026'
 8: '!'

Reconstruction: 'She lives in Göteborg, Sweden in 2026!'


In [5]:
text = "She lives in Göteborg, Sweden in 2026!"

chunks = tokenizer.pattern.findall(text.strip())

print("Original:", text)

for chunk in chunks:
    ids = tokenizer._bpe(chunk)
    tokens = [tokenizer.int_to_token[i] for i in ids]

    print(f"\nChunk: {chunk!r}")
    print("Tokens:", tokens)
    print("IDs:   ", ids)

Original: She lives in Göteborg, Sweden in 2026!

Chunk: 'She'
Tokens: [b'She']
IDs:    [2645]

Chunk: ' lives'
Tokens: [b' lives']
IDs:    [5354]

Chunk: ' in'
Tokens: [b' in']
IDs:    [283]

Chunk: ' Göteborg'
Tokens: [b' G', b'\xc3\xb6', b'te', b'borg']
IDs:    [403, 2867, 593, 13735]

Chunk: ','
Tokens: [b',']
IDs:    [48]

Chunk: ' Sweden'
Tokens: [b' Sweden']
IDs:    [6589]

Chunk: ' in'
Tokens: [b' in']
IDs:    [283]

Chunk: ' 2026'
Tokens: [b' 20', b'26']
IDs:    [588, 5673]

Chunk: '!'
Tokens: [b'!']
IDs:    [37]


In [6]:
words = [
    "the",
    "and",
    "language",
    "transformer",
    "international",
    "electrochemistry",
    "Göteborg",
    "123456789"
]

for word in words:
    ids = tokenizer._bpe(word)
    tokens = [tokenizer.int_to_token[i] for i in ids]

    print(f"{word:<22} -> {tokens}")
    print(f"{'':22}    {len(ids)} BPE tokens\n")

the                    -> [b'the']
                          1 BPE tokens

and                    -> [b'and']
                          1 BPE tokens

language               -> [b'language']
                          1 BPE tokens

transformer            -> [b'trans', b'form', b'er']
                          3 BPE tokens

international          -> [b'in', b'ternational']
                          2 BPE tokens

electrochemistry       -> [b'elect', b'roc', b'hemistry']
                          3 BPE tokens

Göteborg               -> [b'G', b'\xc3\xb6', b'te', b'borg']
                          4 BPE tokens

123456789              -> [b'12', b'34', b'5', b'67', b'89']
                          5 BPE tokens



In [7]:
texts = [
    "This is a test.",
    "She lives in Stockholm.",
    "Natural language processing is interesting.",
    "Göteborg is a city in Sweden."
]

for text in texts:
    encoded = tokenizer(text, padding=False, truncation=False)

    ids = encoded["input_ids"][0]
    decoded = tokenizer.decode(ids)

    print("\nOriginal:", text)
    print("Token IDs:", ids)
    print("Decoded:  ", decoded)

    assert decoded == text.strip()

print("\nAll encoding/decoding tests passed!")


Original: This is a test.
Token IDs: [2, 673, 319, 262, 1890, 50, 3]
Decoded:   This is a test.

Original: She lives in Stockholm.
Token IDs: [2, 2645, 5354, 283, 8475, 14286, 50, 3]
Decoded:   She lives in Stockholm.

Original: Natural language processing is interesting.
Token IDs: [2, 12459, 1281, 5652, 319, 12239, 50, 3]
Decoded:   Natural language processing is interesting.

Original: Göteborg is a city in Sweden.
Token IDs: [2, 75, 2867, 593, 13735, 319, 262, 1015, 283, 6589, 50, 3]
Decoded:   Göteborg is a city in Sweden.

All encoding/decoding tests passed!


In [1]:
from A2_skeleton import A2DecoderLayer, A2ModelConfig

In [2]:
config = A2ModelConfig(vocab_size=100, hidden_size=32, intermediate_size=64, num_attention_heads=4, num_hidden_layers=2,
                       rope_theta=10000, hidden_act='silu', max_position_embeddings=512, rms_norm_eps=1e-5, pad_token_id=0)

In [3]:
A2DecoderLayer(config)

A2DecoderLayer(
  (mha): A2Attention(
    (W_q): Linear(in_features=32, out_features=32, bias=False)
    (W_k): Linear(in_features=32, out_features=32, bias=False)
    (W_v): Linear(in_features=32, out_features=32, bias=False)
    (W_o): Linear(in_features=32, out_features=32, bias=False)
    (norm): RMSNorm((32,), eps=1e-05, elementwise_affine=True)
    (norm_q): RMSNorm((32,), eps=1e-05, elementwise_affine=True)
    (norm_k): RMSNorm((32,), eps=1e-05, elementwise_affine=True)
  )
  (mlp): A2MLP(
    (gated_proj): Linear(in_features=32, out_features=64, bias=False)
    (up_proj): Linear(in_features=32, out_features=64, bias=False)
    (activation): SiLU()
    (down_proj): Linear(in_features=64, out_features=32, bias=False)
  )
  (post_mha_norm): RMSNorm((32,), eps=1e-05, elementwise_affine=True)
  (post_mlp_norm): RMSNorm((32,), eps=1e-05, elementwise_affine=True)
)

In [5]:
# (mlp): Olmo2MLP(
#           (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
#           (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
#           (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
#           (act_fn): SiLUActivation()
#         )

In [14]:
from tokenizer import A1Tokenizer

In [17]:
tokenizer_ours = A1Tokenizer.from_file("tokenizers/tokenizer_16384.pkl")
prompt = "She lives in San"
input_ids = tokenizer_ours(prompt, return_tensors='pt').input_ids.to("cpu")

tensor([[   2, 2645, 5354,  283, 3070]])

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
local_dir = '/data/courses/2025_dat450_dit247/models/OLMo-2-0425-1B'
tokenizer = AutoTokenizer.from_pretrained(local_dir, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(local_dir, local_files_only=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [2]:
model

Olmo2ForCausalLM(
  (model): Olmo2Model(
    (embed_tokens): Embedding(100352, 2048, padding_idx=100277)
    (layers): ModuleList(
      (0-15): 16 x Olmo2DecoderLayer(
        (self_attn): Olmo2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Olmo2RMSNorm((2048,), eps=1e-06)
          (k_norm): Olmo2RMSNorm((2048,), eps=1e-06)
        )
        (mlp): Olmo2MLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (post_attention_layernorm): Olmo2RMSNorm((2048,), eps=1e-06

In [9]:
prompt = "She lives in San"
input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to("cpu")
input_ids

tensor([[8100, 6439,  304, 5960]])

In [18]:
tokenizer.eos_token_id

100257

In [25]:
from torch.distributions import Categorical

def generate(model, tokenizer, prompt, device, max_length, temperature, topk):
    """Generate text given a prompt using top-k sampling."""

    model.eval()

    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)

    stopped_by_eos = False
    eos_token_id = tokenizer.eos_token_id

    with torch.no_grad():
        for _ in range(max_length):
            output = model(input_ids)
            logits = output.logits
            next_token_logits = logits[0, -1, :] / temperature

            top_k_logits, top_k_indices = torch.topk(next_token_logits, topk)

            filtered_logits = torch.full_like(next_token_logits, float('-inf'))

            filtered_logits[top_k_indices] = top_k_logits

            next_token_logits = filtered_logits

            distribution = Categorical(logits=next_token_logits)
            next_token_id = distribution.sample().unsqueeze(0).unsqueeze(0)  # Shape: (1, 1)

            if next_token_id.item() == eos_token_id:
                stopped_by_eos = True
                break

            # input_ids = torch.cat([
            #     input_ids[:, :-1],
            #     next_token_id,
            #     input_ids[:, -1:]
            # ], dim=1)

            input_ids = torch.cat([
                input_ids,
                next_token_id,
            ], dim=1)

    return tokenizer.decode(input_ids[0]), stopped_by_eos

In [26]:
prompts = [
    'She lives in San',
    'Gothenburg is',
    'In natural language processing, a Transformer',
    'Is Stockholm the capital of Sweden? Answer yes or no. The answer is',
    'Write a Python program that reverses a list.'
]

for prompt in prompts:
    text, stopped_by_eos = generate(
        model,
        tokenizer,
        prompt,
        "cpu",
        100,
        0.6,
        20,
    )

    print(f"\nPrompt: {prompt}")
    print(f"Completion: {text}")
    print(
        "Stopped by:",
        "<EOS>" if stopped_by_eos else "token limit"
    )


Prompt: She lives in San
Completion: She lives in San Jose, California.
Stopped by: <EOS>

Prompt: Gothenburg is
Completion: Gothenburg is the city with the highest per capita income in Sweden, with an estimated income of 82,000 SEK ($10,300) per capita, or 8.2 times higher than the national average ($1,100) for Sweden. It is the most expensive city in the world to live in according to Mercer Human Resource Consulting.
Sights.
The city has many well-known sights, including the Royal Palace, the Church of Our Lady, the Royal Theatre and the Museum of Modern Art and
Stopped by: token limit

Prompt: In natural language processing, a Transformer
Completion: In natural language processing, a Transformer is a type of neural network architecture that is capable of learning complex patterns in data by utilizing a series of interconnected “modules.” The architecture was developed by Google Brain Team members Vasileios Nesterov and Jeff Dean and published in 2017.

The Transformer architecture 